In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm

## Cleaning DEM data

In [21]:
dem_raw = pd.read_csv("raw DEM data/dem raw.csv")
dem_raw.head()

,datetime_est,no2,o3
0,9/1/25,NaN,8.8
1,9/1/25 1:00,NaN,9.6
2,9/1/25 2:00,7.4,6.0
3,9/1/25 3:00,5.9,9.6
4,9/1/25 4:00,6.0,8.6


In [22]:
#use mask to isolate midnight times, which by default do not have a 00:00:00 attached
mask = ~dem_raw["datetime_est"].str.contains(":", na=False)
dem_raw.loc[mask, "datetime_est"] = (dem_raw.loc[mask, "datetime_est"] + " 0:00")

dem_raw["datetime_est"] = pd.to_datetime(dem_raw["datetime_est"], format="%m/%d/%y %H:%M")
dem_raw.head(30)

,datetime_est,no2,o3
0,2025-09-01 00:00:00,NaN,8.8
1,2025-09-01 01:00:00,NaN,9.6
2,2025-09-01 02:00:00,7.4,6.0
3,2025-09-01 03:00:00,5.9,9.6
4,2025-09-01 04:00:00,6.0,8.6
5,2025-09-01 05:00:00,6.4,9.6
6,2025-09-01 06:00:00,5.3,10.3
7,2025-09-01 07:00:00,3.9,14.8
8,2025-09-01 08:00:00,3.7,18.6
9,2025-09-01 09:00:00,3.5,21.4


In [23]:
#shift from EST to UTC
#paul confirmed whole time series from this DEM data is firmly EST
dem_raw["datetime_utc"] = pd.to_datetime(dem_raw["datetime_est"]) + pd.Timedelta(hours=5)
dem_raw = dem_raw.drop(columns="datetime_est")
dem_raw = dem_raw[["datetime_utc", "no2", "o3"]]

dem_raw.head(30)

,datetime_utc,no2,o3
0,2025-09-01 05:00:00,NaN,8.8
1,2025-09-01 06:00:00,NaN,9.6
2,2025-09-01 07:00:00,7.4,6.0
3,2025-09-01 08:00:00,5.9,9.6
4,2025-09-01 09:00:00,6.0,8.6
5,2025-09-01 10:00:00,6.4,9.6
6,2025-09-01 11:00:00,5.3,10.3
7,2025-09-01 12:00:00,3.9,14.8
8,2025-09-01 13:00:00,3.7,18.6
9,2025-09-01 14:00:00,3.5,21.4


In [24]:
dem_cleaned = dem_raw
dem_cleaned.to_csv("cleaned DEM data/dem_cleaned.csv", index=False)

## Cleaning quant data

In [25]:
dpw_raw = pd.read_csv("raw Quant data/dpw raw.csv")
mjf_raw = pd.read_csv("raw Quant data/mjf raw.csv")
mjf_recal = pd.read_csv("raw Quant data/mjf recalibrated.csv")
pema_raw = pd.read_csv("raw Quant data/pema raw.csv")
pha_raw = pd.read_csv("raw Quant data/pha raw.csv")

In [26]:
dpw_raw = dpw_raw[["timestamp", "no2", "o3"]].rename(columns={"timestamp": "datetime_utc"})
dpw_raw["datetime_utc"] = pd.to_datetime(dpw_raw["datetime_utc"])

dpw_raw.head(30)

,datetime_utc,no2,o3
0,2026-05-31 23:59:45+00:00,25.082,28.501
1,2026-05-31 23:58:45+00:00,24.654,29.081
2,2026-05-31 23:57:45+00:00,23.764,28.863
3,2026-05-31 23:56:45+00:00,22.344,28.328
4,2026-05-31 23:55:45+00:00,22.344,29.697
5,2026-05-31 23:54:45+00:00,25.490,27.914
6,2026-05-31 23:53:45+00:00,26.753,27.545
7,2026-05-31 23:52:45+00:00,25.143,29.443
8,2026-05-31 23:51:45+00:00,24.745,28.669
9,2026-05-31 23:50:45+00:00,25.224,29.005


In [27]:
dpw_quant_cleaned = dpw_raw.set_index("datetime_utc").resample("h").mean().reset_index()
dpw_quant_cleaned["datetime_utc"] = dpw_quant_cleaned["datetime_utc"].dt.tz_localize(None)

dpw_quant_cleaned["no2"] = dpw_quant_cleaned["no2"].round(1)
dpw_quant_cleaned["o3"] = dpw_quant_cleaned["o3"].round(1)

dpw_quant_cleaned.head()
dpw_quant_cleaned.to_csv("cleaned quant data/dpw_cleaned.csv", index=False)

In [28]:
mjf_raw = mjf_raw[["timestamp", "no2", "o3"]].rename(columns={"timestamp": "datetime_utc"})
mjf_raw["datetime_utc"] = pd.to_datetime(mjf_raw["datetime_utc"])

mjf_raw.head(30)

,datetime_utc,no2,o3
0,2026-05-31 23:59:46+00:00,4.040,43.102
1,2026-05-31 23:58:46+00:00,3.379,45.299
2,2026-05-31 23:57:46+00:00,4.023,43.554
3,2026-05-31 23:56:46+00:00,3.572,44.712
4,2026-05-31 23:55:46+00:00,3.577,45.183
5,2026-05-31 23:54:46+00:00,3.063,45.568
6,2026-05-31 23:53:46+00:00,3.394,44.873
7,2026-05-31 23:52:46+00:00,3.384,44.853
8,2026-05-31 23:51:46+00:00,4.007,44.484
9,2026-05-31 23:50:46+00:00,3.396,45.788


In [29]:
mjf_quant_cleaned1 = mjf_raw.set_index("datetime_utc").resample("h").mean().reset_index()
mjf_quant_cleaned1["datetime_utc"] = mjf_quant_cleaned1["datetime_utc"].dt.tz_localize(None)

mjf_quant_cleaned1.head()

,datetime_utc,no2,o3
0,2026-03-11 00:00:00,8.700633,50.574533
1,2026-03-11 01:00:00,10.013767,44.751100
2,2026-03-11 02:00:00,6.648233,46.229233
3,2026-03-11 03:00:00,6.199550,42.575500
4,2026-03-11 04:00:00,7.638633,37.286617


In [30]:
mjf_recal = mjf_recal[["timestamp_local", "no2", "o3"]]

#some recalibrated timestamps have thousandths of seconds included for some reason. Remove those here
mjf_recal["timestamp_local"] = mjf_recal["timestamp_local"].astype(str).str.replace(r"\.000$", "", regex=True)
mjf_recal["timestamp_local"] = mjf_recal["timestamp_local"].astype(str).str.replace(r"\.500$", "", regex=True)

mjf_recal["timestamp_local"] = pd.to_datetime(mjf_recal["timestamp_local"])

#needed to manually include timezone column because quant only gave me local timestamp for recalibrated data
mjf_recal.to_csv("raw Quant data/mjf_to_add_tz.csv", index=False)

In [31]:
#manually input timezone each reading took place at
mjf_recal_tz = pd.read_csv("raw Quant data/mjf_recal_tz.csv")
mjf_recal_tz.head()

,timestamp_local,no2,o3,tz
0,8/28/25 14:16,-5.594733,235.480795,edt
1,8/28/25 14:17,-5.594733,214.030878,edt
2,8/28/25 14:18,-5.594733,195.068412,edt
3,8/28/25 14:19,-5.594732,-261.291336,edt
4,8/28/25 14:20,-5.594324,61.186989,edt


In [32]:
mjf_recal_tz["timestamp_local"] = pd.to_datetime(mjf_recal_tz["timestamp_local"])

#adjust to utc based on designated timezone
mjf_recal_tz["datetime_utc"] = mjf_recal_tz["timestamp_local"] + pd.to_timedelta(mjf_recal_tz["tz"].str.lower().map({
            "edt": 4,
            "est": 5}),
            unit="h")

mjf_recal_tz = mjf_recal_tz[["datetime_utc", "no2", "o3"]]
mjf_recal_tz.head(30)

/var/folders/cv/16s476bd69z2kjqnmm06ls5c0000gn/T/ipykernel_46762/1684752030.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mjf_recal_tz["timestamp_local"] = pd.to_datetime(mjf_recal_tz["timestamp_local"])


,datetime_utc,no2,o3
0,2025-08-28 18:16:00,-5.594733,235.480795
1,2025-08-28 18:17:00,-5.594733,214.030878
2,2025-08-28 18:18:00,-5.594733,195.068412
3,2025-08-28 18:19:00,-5.594732,-261.291336
4,2025-08-28 18:20:00,-5.594324,61.186989
5,2025-08-28 18:21:00,-5.593890,102.366522
6,2025-08-28 18:22:00,-5.591060,105.408751
7,2025-08-28 18:23:00,-5.565535,102.416938
8,2025-08-28 18:24:00,-5.089504,99.235495
9,2025-08-28 18:25:00,-4.854538,94.138863


In [33]:
mjf_quant_cleaned2 = mjf_recal_tz.set_index("datetime_utc").resample("h").mean().reset_index()
mjf_quant_cleaned2["datetime_utc"] = mjf_quant_cleaned2["datetime_utc"].dt.tz_localize(None)

mjf_quant_cleaned2.head()

,datetime_utc,no2,o3
0,2025-08-28 18:00:00,-1.744937,73.771410
1,2025-08-28 19:00:00,2.202533,57.932656
2,2025-08-28 20:00:00,2.432583,56.728456
3,2025-08-28 21:00:00,2.577775,56.368741
4,2025-08-28 22:00:00,3.062726,57.325889


In [34]:
mjf_quant_cleaned = pd.concat([mjf_quant_cleaned2, mjf_quant_cleaned1], ignore_index=True)

#can't merge directly because there are slight floating point differences between concentrations between recalibrated data and data from the portal
#ensure preference of web portal data here so no merge issues (tho almost no difference anyways)
mjf_quant_cleaned = mjf_quant_cleaned.drop_duplicates(subset="datetime_utc", keep="last").sort_values("datetime_utc").reset_index(drop=True)

mjf_quant_cleaned["no2"] = mjf_quant_cleaned["no2"].round(1)
mjf_quant_cleaned["o3"] = mjf_quant_cleaned["o3"].round(1)

mjf_quant_cleaned.to_csv("cleaned quant data/mjf_cleaned.csv", index=False)

mjf_quant_cleaned.head()

,datetime_utc,no2,o3
0,2025-08-28 18:00:00,-1.7,73.8
1,2025-08-28 19:00:00,2.2,57.9
2,2025-08-28 20:00:00,2.4,56.7
3,2025-08-28 21:00:00,2.6,56.4
4,2025-08-28 22:00:00,3.1,57.3


In [35]:
pema_raw = pema_raw[["timestamp", "no2", "o3"]].rename(columns={"timestamp": "datetime_utc"})
pema_raw["datetime_utc"] = pd.to_datetime(pema_raw["datetime_utc"])

pema_raw.head(30)

,datetime_utc,no2,o3
0,2026-05-31 23:59:22+00:00,16.724,21.874
1,2026-05-31 23:58:22+00:00,16.719,21.425
2,2026-05-31 23:57:22+00:00,17.292,22.222
3,2026-05-31 23:56:22+00:00,16.697,21.455
4,2026-05-31 23:55:22+00:00,16.609,23.275
5,2026-05-31 23:54:22+00:00,16.027,22.950
6,2026-05-31 23:53:22+00:00,17.187,21.327
7,2026-05-31 23:52:22+00:00,16.622,21.436
8,2026-05-31 23:51:22+00:00,18.398,21.029
9,2026-05-31 23:50:22+00:00,17.214,21.746


In [36]:
pema_quant_cleaned = (pema_raw.set_index("datetime_utc").resample("h").mean().reset_index())
pema_quant_cleaned["datetime_utc"] = pema_quant_cleaned["datetime_utc"].dt.tz_localize(None)

pema_quant_cleaned["no2"] = pema_quant_cleaned["no2"].round(1)
pema_quant_cleaned["o3"] = pema_quant_cleaned["o3"].round(1)

pema_quant_cleaned.head()
pema_quant_cleaned.to_csv("cleaned quant data/pema_cleaned.csv", index=False)

In [37]:
pha_raw = pha_raw[["timestamp", "no2", "o3"]].rename(columns={"timestamp": "datetime_utc"})
pha_raw["datetime_utc"] = pd.to_datetime(pha_raw["datetime_utc"])

pha_raw.head(30)

,datetime_utc,no2,o3
0,2026-05-31 23:59:26+00:00,22.540,14.699
1,2026-05-31 23:58:26+00:00,23.036,15.473
2,2026-05-31 23:57:26+00:00,23.073,14.127
3,2026-05-31 23:56:26+00:00,21.967,15.263
4,2026-05-31 23:55:26+00:00,20.787,15.539
5,2026-05-31 23:54:26+00:00,23.023,15.466
6,2026-05-31 23:53:26+00:00,25.336,13.952
7,2026-05-31 23:52:26+00:00,24.520,14.228
8,2026-05-31 23:51:26+00:00,24.976,14.093
9,2026-05-31 23:50:26+00:00,23.648,14.956


In [38]:
pha_quant_cleaned = pha_raw.set_index("datetime_utc").resample("h").mean().reset_index()
pha_quant_cleaned["datetime_utc"] = pha_quant_cleaned["datetime_utc"].dt.tz_localize(None)

pha_quant_cleaned["no2"] = pha_quant_cleaned["no2"].round(1)
pha_quant_cleaned["o3"] = pha_quant_cleaned["o3"].round(1)

pha_quant_cleaned.head()
pha_quant_cleaned.to_csv("cleaned quant data/pha_cleaned.csv", index=False)